# 02 - Preparação dos Dados | Case Olist

Tratamento de outliers/nulos e modelagem dimensional. A ideia é transformar as
9 tabelas soltas em uma tabela fato (pedidos, no nível de item) e tabelas
dimensão (clientes, produtos, vendedores, tempo, geolocalização), prontas para
as análises de Crescimento e de Logística usarem.

Este notebook depende do `data/base/` já estar com os 9 CSVs do Kaggle.

In [1]:
import pandas as pd

## 1. Carregando os dados

In [2]:
customers = pd.read_csv("../data/base/olist_customers_dataset.csv")
orders = pd.read_csv("../data/base/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/base/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/base/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/base/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/base/olist_products_dataset.csv")
sellers = pd.read_csv("../data/base/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../data/base/olist_geolocation_dataset.csv")
category_translation = pd.read_csv("../data/base/product_category_name_translation.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

print("Todas as tabelas foram carregadas!")

Todas as tabelas foram carregadas!


## 2. Verificando outliers em price e freight_value

Aqui só olhamos os números, sem apagar nada. A decisão de remover ou não um valor deve ser combinada com o grupo, porque afeta as análises de Crescimento e de Logística.

In [3]:
print(order_items["price"].describe())

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64


In [4]:
print(order_items["freight_value"].describe())

count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64


O preço vai de R 0,85 até R 6.735,00, com metade dos itens custando até R 74,99. O valor máximo é bem mais alto que a média, mas pode ser um produto caro de verdade (ex.: eletrônico), não necessariamente um erro. Mesma lógica para o frete, que vai até R 409,68. Por enquanto, só documentamos — nenhum valor foi removido.

## 3. Verificando outliers e nulos em product_weight_g

In [5]:
print(products["product_weight_g"].describe())

count    32949.000000
mean      2276.472488
std       4282.038731
min          0.000000
25%        300.000000
50%        700.000000
75%       1900.000000
max      40425.000000
Name: product_weight_g, dtype: float64


In [6]:
print("Produtos com peso nulo:", products["product_weight_g"].isnull().sum())
print("Produtos com peso 0:", (products["product_weight_g"] == 0).sum())

Produtos com peso nulo: 2
Produtos com peso 0: 4


Só 2 produtos com peso nulo e 4 com peso 0, de quase 33 mil produtos — impacto muito pequeno para a análise. Vale marcar esses 6 produtos como pendentes de revisão, mas não precisa tratar agora.

## 4. Confirmando o tratamento de order_delivered_customer_date

Já vimos no notebook anterior que esse campo fica nulo quando o pedido ainda não foi entregue — não é erro. Regra adotada: manter os nulos como estão (não preencher com data falsa), e excluir esses pedidos de qualquer cálculo de lead time ou atraso.

## 5. Montando a tabela fato (fato_pedidos)

Grão da tabela: um item de pedido por linha (mesmo grão de `order_items`). É o nível mais detalhado, então dá pra somar/agrupar do jeito que cada análise precisar depois.

In [7]:
fato_pedidos = order_items.merge(
    orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp",
            "order_delivered_customer_date", "order_estimated_delivery_date"]],
    on="order_id",
    how="left",
)

fato_pedidos["atrasado"] = (
    fato_pedidos["order_delivered_customer_date"] > fato_pedidos["order_estimated_delivery_date"]
)

print(fato_pedidos.shape)
fato_pedidos.head()

(112650, 13)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,atrasado
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,False


`payment_value` fica de fora da tabela fato porque `order_payments` está no nível de pedido, não de item — um pedido com 3 itens tem 1 valor de pagamento só. Juntar direto duplicaria o valor pago em cada item. Quando precisar do valor pago, use `order_payments` agrupado por `order_id` separadamente.

## 6. Montando as tabelas dimensão

### dim_clientes

In [8]:
dim_clientes = customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]]
print(dim_clientes.shape)
dim_clientes.head()

(99441, 4)


,customer_id,customer_unique_id,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


### dim_produtos

In [9]:
dim_produtos = products.merge(category_translation, on="product_category_name", how="left")
dim_produtos = dim_produtos[["product_id", "product_category_name", "product_category_name_english",
                              "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]]
print(dim_produtos.shape)
dim_produtos.head()

(32951, 7)


,product_id,product_category_name,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,625.0,20.0,17.0,13.0


### dim_vendedores

In [10]:
dim_vendedores = sellers[["seller_id", "seller_city", "seller_state"]]
print(dim_vendedores.shape)
dim_vendedores.head()

(3095, 3)


,seller_id,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,braganca paulista,SP


### dim_tempo

Uma linha por data de compra, com ano, mês e dia separados — facilita filtrar e agrupar nos gráficos de evolução.

In [11]:
dim_tempo = pd.DataFrame({"data": orders["order_purchase_timestamp"].dt.date.unique()})
dim_tempo["data"] = pd.to_datetime(dim_tempo["data"])
dim_tempo["ano"] = dim_tempo["data"].dt.year
dim_tempo["mes"] = dim_tempo["data"].dt.month
dim_tempo["dia"] = dim_tempo["data"].dt.day
dim_tempo["ano_mes"] = dim_tempo["data"].dt.to_period("M").astype(str)
dim_tempo = dim_tempo.sort_values("data").reset_index(drop=True)

print(dim_tempo.shape)
dim_tempo.head()

(634, 5)


,data,ano,mes,dia,ano_mes
0,2016-09-04,2016,9,4,2016-09
1,2016-09-05,2016,9,5,2016-09
2,2016-09-13,2016,9,13,2016-09
3,2016-09-15,2016,9,15,2016-09
4,2016-10-02,2016,10,2,2016-10


### dim_geolocalizacao

A tabela original tem várias linhas para o mesmo `zip_code_prefix` (vários pontos de coordenada por CEP). Para virar uma dimensão de verdade (uma linha por CEP), agrupamos e tiramos a média de latitude/longitude.

In [12]:
dim_geolocalizacao = geolocation.groupby("geolocation_zip_code_prefix").agg(
    geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", "first"),
    geolocation_state=("geolocation_state", "first"),
).reset_index()

print(dim_geolocalizacao.shape)
dim_geolocalizacao.head()

(19015, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


## 7. Salvando as tabelas em data/processado/

In [13]:
fato_pedidos.to_csv("../data/processado/fato_pedidos.csv", index=False)
dim_clientes.to_csv("../data/processado/dim_clientes.csv", index=False)
dim_produtos.to_csv("../data/processado/dim_produtos.csv", index=False)
dim_vendedores.to_csv("../data/processado/dim_vendedores.csv", index=False)
dim_tempo.to_csv("../data/processado/dim_tempo.csv", index=False)
dim_geolocalizacao.to_csv("../data/processado/dim_geolocalizacao.csv", index=False)

print("Tabelas salvas em data/processado/")

Tabelas salvas em data/processado/
